In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get('OPENAI_API_KEY')

In [2]:
import fitz
import json
import pandas as pd
from pptx import Presentation
from PIL import Image
import pytesseract
from unstructured.partition.auto import partition

# ---------- PDF Parser ----------
def pdf_parser(file):
    doc = fitz.open(file)

    text = ""

    for page in doc:
        text = text + page.get_text()

    return {
        "text": text,
        "metadata": {
            "file_name": file.name
        }
    }

# ---------- CSV Parser ----------
def csv_parser(filepath):

    df = pd.read_csv(filepath)

    return {
        "text": df.to_string(index=False),
        "metadata": {
            "file_name": filepath.name
        }
    }

# ---------- Image Parser ----------
def image_parser(filepath):

    img = Image.open(filepath)
    text = pytesseract.image_to_string(image=img)

    return {
        "text": text,
        "metadata": {
            "file_name": filepath.name
        }
    }

# ---------- JSON Parser ----------
def json_parser(filepath):

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    return {
        "text": json.dumps(data, indent=2),
        "metadata": {
            "file_name":filepath.name
        }
    }

# ---------- PPT Parser ----------
def ppt_parser(filepath):
    elements = partition(filename=str(filepath))

    text = "\n".join(
        element.text.strip()
        for element in elements
        if hasattr(element, "text") and element.text
    )

    return {
        "text": text,
        "metadata": {
            "file_name": filepath.name
        }
    }

In [3]:
from pathlib import Path
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

UPLOAD_DIR = Path('./data')

all_files = []

for i in UPLOAD_DIR.rglob("*"):
    if i.is_file():
        all_files.append(i)

print("Length of all files is:", len(all_files))

print(all_files)

PARSERS = {
    ".pdf": pdf_parser,
    ".json": json_parser,
    ".csv": csv_parser,
    ".ppt": ppt_parser,
    ".pptx": ppt_parser,
    ".png": image_parser,
    ".jpg": image_parser,
    ".jpeg": image_parser,
}

documents = []

for file in all_files:

    print(file)
    extension = file.suffix.lower()
    
    parser = PARSERS.get(extension)

    if parser is None:
        print(f"Skipping {file.name}")
        continue

    doc = parser(file)
    documents.append(doc)

documents

Length of all files is: 5
[WindowsPath('data/csv/contact_center_service_automation_synthetic.csv'), WindowsPath('data/image/ml.png'), WindowsPath('data/json/contact_center_service_automation_synthetic.json'), WindowsPath('data/pdf/NIPS-2017-attention-is-all-you-need-Paper.pdf'), WindowsPath('data/ppt/sample-1.pptx')]
data\csv\contact_center_service_automation_synthetic.csv
data\image\ml.png
data\json\contact_center_service_automation_synthetic.json
data\pdf\NIPS-2017-attention-is-all-you-need-Paper.pdf
data\ppt\sample-1.pptx


[{'text': 'ticket_id customer_id          created_at         resolved_at  channel language   product     issue_type priority agent_id  wait_time_sec  handle_time_sec  first_call_resolution  csat_score customer_sentiment      status  knowledge_article_used  llm_confidence  escalated region\nTKT000001   CUST28289 2026-04-27 09:41:00 2026-04-27 09:47:18    Voice    Hindi Broadband Password Reset      Low    AG103             30              348                      1         3.1           Positive   Escalated                       0            0.84          1  North\nTKT000002   CUST54597 2026-03-23 18:37:00 2026-03-23 18:48:40     Chat    Tamil  Internet  Network Issue Critical    AG104             11              689                      0         3.4            Neutral     Pending                       1            0.92          1  South\nTKT000003   CUST91070 2026-06-27 03:36:00 2026-06-27 03:52:20    Email    Hindi    Mobile  Network Issue      Low    AG122             85            

In [4]:
type(documents)

list

## **Chunking**

### **Text Splitter**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 600,
    chunk_overlap= 100,
    separators= ["\n\n", "\n", ". ", " ", ""]
)

### **Create Chunks**

In [6]:
from langchain_core.documents import Document

chunked_documents = []

for doc in documents:

    text = doc["text"]
    metadata = doc["metadata"]

    chunks = text_splitter.split_text(text)

    for idx, chunk in enumerate(chunks):
        chunk_metadata = {
            'chunk_id': idx,
            **metadata
        }
        chunked_documents.append(Document(page_content = chunk, metadata = chunk_metadata))

chunked_documents[0:2]


[Document(metadata={'chunk_id': 0, 'file_name': 'contact_center_service_automation_synthetic.csv'}, page_content='ticket_id customer_id          created_at         resolved_at  channel language   product     issue_type priority agent_id  wait_time_sec  handle_time_sec  first_call_resolution  csat_score customer_sentiment      status  knowledge_article_used  llm_confidence  escalated region\nTKT000001   CUST28289 2026-04-27 09:41:00 2026-04-27 09:47:18    Voice    Hindi Broadband Password Reset      Low    AG103             30              348                      1         3.1           Positive   Escalated                       0            0.84          1  North'),
 Document(metadata={'chunk_id': 1, 'file_name': 'contact_center_service_automation_synthetic.csv'}, page_content='TKT000002   CUST54597 2026-03-23 18:37:00 2026-03-23 18:48:40     Chat    Tamil  Internet  Network Issue Critical    AG104             11              689                      0         3.4            Neutral  

## **Vector DB**

In [7]:
from pymilvus import MilvusClient, DataType

endpoint = os.environ.get("ZILLIZ_URI")
milvus_api_key = os.environ.get("ZILLIZ_TOKEN")
collection_name = "my_agentic_rag"

client = MilvusClient(
    uri= endpoint,
    token= milvus_api_key
)

schema = client.create_schema(
    auto_id=False,
    enable_dynamic_field=True
)

schema.add_field(
    field_name="id",
    datatype=DataType.VARCHAR,
    is_primary=True,
    max_length=36,
)

schema.add_field(
    field_name="vector",
    datatype=DataType.FLOAT_VECTOR,
    dim=1536,
)

schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=65535,
)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 36}, 'is_primary': True, 'auto_id': False}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}], 'enable_dynamic_field': True, 'enable_namespace': False}

### **Create Collection**

In [8]:
DIMENSION = 1536

if not client.has_collection(collection_name):

    client.create_collection(
        collection_name=collection_name,
        dimension=DIMENSION,
        metric_type="COSINE",
        schema=schema
    )

print("Collection Ready")

Collection Ready


## **Embedding Model**

In [9]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)

### **Prepare chunks and embeddings**

In [10]:
texts = []

for doc in chunked_documents:
    texts.append(doc.page_content)

embeddings = embedding_model.embed_documents(texts)

print(len(chunked_documents))
print(len(embeddings))

print("Embedding Dimension:", len(embeddings[0]))

908
908
Embedding Dimension: 1536


### **Create Records**

In [11]:
import uuid

records = []

for doc, embedding in zip(chunked_documents, embeddings):
    record = {
        "id": str(uuid.uuid4()),
        "vector": embedding,
        "text": doc.page_content,
        **doc.metadata
    }

    records.append(record)

print(records[0])

{'id': '49ec7878-0091-48bb-ac2b-b3d7c96a1090', 'vector': [-0.00579833984375, -0.03582763671875, 0.0270843505859375, -0.0099945068359375, -0.051055908203125, -0.01434326171875, 0.004180908203125, 0.0284576416015625, 0.005260467529296875, 0.01555633544921875, 0.04052734375, 0.0003669261932373047, -0.0310211181640625, 0.004131317138671875, 0.04718017578125, 0.007305145263671875, 0.0080413818359375, -0.015625, -0.07470703125, 0.0188751220703125, 0.019073486328125, 0.0243682861328125, -0.01203155517578125, 0.00472259521484375, -0.023773193359375, 0.0172882080078125, -0.0222625732421875, -0.029327392578125, 0.02825927734375, -0.037811279296875, 0.046173095703125, -0.036376953125, 0.0038433074951171875, 0.002262115478515625, -0.004749298095703125, 0.057708740234375, -0.028472900390625, 0.013885498046875, 0.0268402099609375, -0.029327392578125, -0.039154052734375, -0.0225372314453125, -0.036407470703125, 0.042938232421875, -0.045196533203125, -0.01457977294921875, -0.006534576416015625, 0.0136

### **Insert into Milvus**

In [12]:
client.insert(
    collection_name=collection_name,
    data=records
)

{'insert_count': 908, 'ids': ['49ec7878-0091-48bb-ac2b-b3d7c96a1090', 'cf010519-5001-4ea7-882c-a2842dc6e5d2', 'd7f53ef9-7f0c-4570-bbba-879e98bb7846', 'c3bb9d1f-5ee6-4fee-a6c9-d0fee54c3644', 'd9346196-da3f-4041-94b5-af8dcab0bfd7', '8050ec09-5733-47b5-9306-8a6fb9661d9c', '9c0d8af7-797b-4416-bb80-5909be3ada1c', 'cfb90121-459c-4c30-8f84-fbe4a3653e67', 'e0f30cce-afe0-4583-bcd0-15bfff62661c', 'de8484e7-5819-4c71-86cf-0ca13f80b1ff', '3a530b07-2078-4fd8-8c70-14751d391a50', 'd8976964-8c80-4860-bcb5-7b5b250f6703', '1de5a48e-1335-420a-a478-438f60e770c1', '3fd36ff5-f826-47ee-bb4b-a5ecb89f6d43', '61f382f0-bec2-43f6-856b-0eb582a86976', 'f7b2df0a-dc3f-40c7-92a7-335474978a6a', '175be02f-3b5a-4e35-bd04-56cdb6db5e40', 'cee1574b-b6e7-4999-9798-e2bcfb82ed2d', 'a11a3686-5587-416b-8030-a6b134506b00', '374c237e-9425-4730-8b17-f3270d0b3968', 'd776a9fc-9090-4088-b1fa-65eabb893930', '0a6e32bd-14ad-479f-af43-bb9a6efeed4a', '1d2291b8-8e49-4904-8382-79dc4772a65a', '64da6b07-74fa-48f9-933d-006d99129dc7', '48203c26-